# Visualizing sentence graphs and their optimal transport

1. Build the graph of a tag-augmented transcript and draw it on a timeline: one lane per modality, one node per cue, a bar for its extent in words, and links between cues that overlap in time ($C = 0$).
2. Show its structure matrix $C$ (number of words between cues).
3. Compute the FGW distance between two graphs and draw the optimal coupling $\pi$: links between matched cues, with a width proportional to the transported mass.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from graph_multimodalsentence import (Graph, draw_coupling, draw_coupling_matrix,
                                      draw_sentence_graph, draw_structure_matrix)
from ot_distances import Fused_Gromov_Wasserstein_distance

embedding = SentenceTransformer("sentence-transformers/bert-base-nli-mean-tokens")
ALPHA = 0.14            # trade-off between feature and structure costs
FIGURE_DIR = "figures"  # set to None to skip saving
if FIGURE_DIR:
    os.makedirs(FIGURE_DIR, exist_ok=True)


def save(name):
    if FIGURE_DIR:
        plt.savefig(os.path.join(FIGURE_DIR, name), dpi=300, bbox_inches="tight")

## Sentences

Two transcripts from the data, or write your own.

In [ ]:
data = pd.read_csv("../data_with_text_tag/generated_instructions_with_intention_explanation_"
                   "full_ability_gender_neutral_gpt.csv")
sentence1 = data["instruction_with_intention"].iloc[0]
sentence2 = data["instruction_with_intention"].iloc[5]
# sentence1 = "{f: smile} Hello [excited] how are {g: Wave} you {t: Second.Person} today"

g1 = Graph.from_sentence(sentence1, embedding)
g2 = Graph.from_sentence(sentence2, embedding)
print(sentence1, "\n", sentence2, sep="\n")

## Cues of a graph

In [ ]:
def cue_table(graph):
    return pd.DataFrame([{"node": k, "label": cue.label, "onset": cue.onset, "duration": cue.duration,
                          "description": cue.description} for k, cue in enumerate(graph.cues)])


cue_table(g1)

## Graph on a timeline

In [ ]:
draw_sentence_graph(g1, title="Graph 1")
save("graph1.png")
plt.show()

draw_sentence_graph(g2, title="Graph 2")
save("graph2.png")
plt.show()

## Structure matrix

In [ ]:
draw_structure_matrix(g1)
save("structure_matrix1.png")
plt.show()

## FGW distance and optimal coupling

`thresh` hides links carrying less than this fraction of the largest mass.

In [ ]:
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")
distance = fgw.graph_d(g1, g2)

draw_coupling(g1, g2, fgw.transp, thresh=0.1, distance=distance)
save("coupling.png")
plt.show()

draw_coupling_matrix(g1, g2, fgw.transp)
save("coupling_matrix.png")
plt.show()

## Effect of $\alpha$

With $\alpha = 0$ only the cue descriptions are matched (Wasserstein); with $\alpha = 1$ only the temporal structure (Gromov-Wasserstein).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 21))
for ax, alpha in zip(axes, [0, ALPHA, 1]):
    fgw = Fused_Gromov_Wasserstein_distance(alpha=alpha, features_metric="sqeuclidean", method="temporal")
    distance = fgw.graph_d(g1, g2)
    draw_coupling(g1, g2, fgw.transp, ax=ax, thresh=0.1)
    ax.set_title(rf"$\alpha$ = {alpha}: FGW distance = {distance:.4f}", fontsize=11, pad=18)
fig.tight_layout()
save("coupling_alphas.png")
plt.show()

## Animation

Word-by-word GIF of the active cues of a graph.

In [ ]:
g1.create_sentence_gif(os.path.join(FIGURE_DIR or ".", "graph1.gif"))